# Task 1.1 - Read the file the foundations way

Read the file and print header and row count

In [ ]:
import csv
with open("/content/qDCCmpyfss1kBJJB.csv","r",newline="") as file:
  reader= csv.DictReader(file)
  header =reader.fieldnames
  print(header)

  count =0
  for row in reader:
    count+=1

  print(f"Number of rows: {count}")

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Number of rows: 1020


Data type conversion

In [ ]:
# Write a function parse_row(row) that converts age, fee_inr and prior_no_shows to int and distance_km to float, returning the converted dictionary.
def parse_row(row):
    parsed = row.copy()
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed

accepted_list = []
rejected_list = []

with open("/content/qDCCmpyfss1kBJJB.csv", "r", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        try:
            parsed_row = parse_row(row)
            accepted_list.append(parsed_row)
        except ValueError:
            rejected_list.append(row)

print(f"Accepted rows: {len(accepted_list)}")
print(f"Rejected rows: {len(rejected_list)}")

Accepted rows: 988
Rejected rows: 32


Dumping rejected row in json file

In [ ]:
import json

with open("rejected_rows.json", "w") as file:
    json.dump(rejected_list, file, indent=2)

# Task 1.2 — Inspect and clean, pass 1 *italicized text*

In [ ]:
import pandas as pd

data = pd.read_csv("/content/qDCCmpyfss1kBJJB.csv")

data.shape

(1020, 14)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB


In [ ]:
data.isnull().sum()

,0
appointment_id,0
patient_id,0
clinic_id,0
booked_on,0
appt_date,1
age,12
gender,0
distance_km,20
fee_inr,0
booking_channel,0


Null values in each column: appt_date - 1, age - 12, distance_km - 20, status - 4

In [ ]:
data.duplicated().sum()

np.int64(20)

In [ ]:
type(data)

pandas.core.frame.DataFrame

20 rows are duplicated in the data

In [ ]:
import pandas as pd

# (a) Drop exact duplicate rows
data = data.drop_duplicates()
print(f"Row count after dropping duplicates: {len(data)}")

# (b) Clean booking_channel with .str.strip().str.title()
data["booking_channel"] = data["booking_channel"].str.strip().str.title()
print(f"Row count after cleaning booking_channel: {len(data)}")

# (c) Convert appt_date and booked_on to datetime
data["appt_date"] = pd.to_datetime(data["appt_date"], errors="coerce")
data["booked_on"] = pd.to_datetime(data["booked_on"], errors="coerce")
print(f"Row count after datetime conversion: {len(data)}")

# (d) Drop rows where appt_date is missing after conversion or status is missing
data = data.dropna(subset=["appt_date", "status"])
print(f"Final row count after dropping missing appt_date/status: {len(data)}")

Row count after dropping duplicates: 1000
Row count after cleaning booking_channel: 1000
Row count after datetime conversion: 1000
Final row count after dropping missing appt_date/status: 990


# Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
import numpy as np

# (a) Treat any age below 1 or above 100 as invalid and set it to missing
invalid_age_mask = (data["age"] < 1) | (data["age"] > 100)
data.loc[invalid_age_mask, "age"] = np.nan
data

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-In,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-In,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended
5,A0005,P251,C01,2024-11-23,2025-01-02,64.0,F,19.0,700,Phone,Consultation,No,0,No-Show
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1015,A0996,P415,C05,2025-01-09,2025-01-31,20.0,M,13.0,700,Phone,Consultation,No,1,Attended
1016,A0997,P247,C05,2025-02-23,2025-02-27,77.0,M,17.4,400,App,Follow-up,Yes,3,No-Show
1017,A0998,P422,C06,2025-05-05,2025-05-16,58.0,M,3.2,850,Walk-In,Consultation,No,1,Attended
1018,A0999,P509,C05,2025-03-30,2025-04-12,59.0,F,180.0,300,Walk-In,Follow-up,Yes,0,Attended


In [ ]:
# (b) Fill missing age with the median (then .round().astype(int)) and missing distance_km with its median.

data["age"].isnull().sum()


np.int64(17)

In [ ]:
data["age"]=data["age"].fillna(data["age"].median())
data["age"].isnull().sum()

np.int64(0)

In [ ]:
distance_median = data["distance_km"].median()
data["distance_km"] = data["distance_km"].fillna(distance_median)

# (c) Compute Q1, Q3, and IQR, and properly clip values using np.clip(array, a_min, a_max)

# Distance
Q1_dist = data["distance_km"].quantile(0.25)
Q3_dist = data["distance_km"].quantile(0.75)
IQR_dist = Q3_dist - Q1_dist
low_dist = Q1_dist - 1.5 * IQR_dist
up_dist = Q3_dist + 1.5 * IQR_dist

# Fee
Q1_fee = data["fee_inr"].quantile(0.25)
Q3_fee = data["fee_inr"].quantile(0.75)
IQR_fee = Q3_fee - Q1_fee
low_fee = Q1_fee - 1.5 * IQR_fee
up_fee = Q3_fee + 1.5 * IQR_fee

# Count how many values will be capped
capped_dist = ((data["distance_km"] < low_dist) | (data["distance_km"] > up_dist)).sum()
capped_fee = ((data["fee_inr"] < low_fee) | (data["fee_inr"] > up_fee)).sum()

data["distance_km"] = np.clip(data["distance_km"], low_dist, up_dist)
data["fee_inr"] = np.clip(data["fee_inr"], low_fee, up_fee)

data["distance_km"] = data["distance_km"].round(2)
data["fee_inr"] = data["fee_inr"].round().astype(int)

print(f"Capped values in 'distance_km': {capped_dist}; Upper bound : {up_dist:.2f}; Lower Bound : {low_dist:.2f}")
print(f"Capped values in 'fee_inr': {capped_fee}; Upper bound : {up_fee:.2f}; Lower Bound : {low_fee:.2f}")

Capped values in 'distance_km': 26; Upper bound : 25.56; Lower Bound : -7.94
Capped values in 'fee_inr': 3; Upper bound : 1218.75; Lower Bound : 68.75


# Task 1.4 — Features and a NumPy summary

In [ ]:
# Create lead_days = days between booked_on and appt_date (use .dt.days) and no_show = 1 if status is "No-Show" else 0.
# Then take X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy() and, without loops, compute:

data["lead_days"] = (data["appt_date"] - data["booked_on"]).dt.days

print(f"Range of lead days: {data['lead_days'].max() - data['lead_days'].min()}")
X = data[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) the mean of each column (axis=0)
mean_X = X.mean(axis=0)
print(f"Mean of age, distance, fee, lead_days are, respectively: {mean_X}")

# (b) the number and share of rows with lead_days > 14 using a boolean mask
mask_lead_days = X[:, 3] > 14
num_lead_days_gt_14 = mask_lead_days.sum()
share_lead_days_gt_14 = mask_lead_days.mean()
print(f"Number of lead days > 14: {num_lead_days_gt_14}")
print(f"Share of lead days > 14: {share_lead_days_gt_14:.4f}")

# (c) z-scores of the distance_km column using (x - np.mean(x)) / np.std(x) and the count of rows with |z| > 2.
distances = X[:, 1]
dist_mean = np.mean(distances)
dist_std = np.std(distances)

# Calculate z-scores
z_scores = (distances - dist_mean) / dist_std

# Count rows with absolute z-score > 2
outliers_mask = np.abs(z_scores) > 2
outliers_count = outliers_mask.sum()

print(f"Number of rows with |z| > 2 in distance_km: {outliers_count}")

Range of lead days: 44
Mean of age, distance, fee, lead_days are, respectively: [ 48.17373737   9.50086869 635.71414141  11.81919192]
Number of lead days > 14: 296
Share of lead days > 14: 0.2990
Number of rows with |z| > 2 in distance_km: 53


# Task 1.5 — Enrich the clinics from a live API

In [ ]:
clinics = pd.read_csv("/content/jbFzPdd2gB2Ex2TY.csv")

import requests

temps = []

for _, row in clinics.iterrows():
    temp = None
    reason = ""

    for attempt in range(2):  # first try + one retry
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={"latitude": row["latitude"], "longitude": row["longitude"],
                        "hourly": "temperature_2m", "forecast_days": 1},
                timeout=10,
            )
            if response.status_code == 200:
                temp = response.json()["hourly"]["temperature_2m"][0]
                break
            reason = f"status code {response.status_code}"

        except requests.exceptions.RequestException as e:
            reason = str(e)
        except KeyError:
            reason = "hourly data missing"

    if temp is None:
        print(row["clinic_name"], "-", reason)  # change to your name column
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)

# Task 1.6 — Export the clean table and write the SQL load script

Sort the cleaned data by appointment_id, format both date columns as YYYY-MM-DD text (.dt.strftime("%Y-%m-%d")), and save appointments_clean.csv with exactly these 16 columns, in this order: appointment_id, patient_id, clinic_id, booked_on, appt_date, age, gender, distance_km, fee_inr, booking_channel, visit_type, reminder_sent, prior_no_shows, status, lead_days, no_show.

In [ ]:
import pandas as pd

# 1. Sort the cleaned data by appointment_id
data = data.sort_values(by="appointment_id")

# 2. Format both date columns as YYYY-MM-DD text
data["booked_on"] = data["booked_on"].dt.strftime("%Y-%m-%d")
data["appt_date"] = data["appt_date"].dt.strftime("%Y-%m-%d")

# 3. Create the no_show column (1 if status is "No-Show" else 0)
data["no_show"] = (data["status"] == "No-Show").astype(int)

# 4. Save with exactly these 16 columns in this order
columns_order = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"
]
data_clean = data[columns_order]

# 5. Save to appointments_clean.csv
data_clean.to_csv("appointments_clean.csv", index=False)

# Display tail of the cleaned dataframe
data_clean.tail()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
1015,A0996,P415,C05,2025-01-09,2025-01-31,20.0,M,13.00,700,Phone,Consultation,No,1,Attended,22,0
1016,A0997,P247,C05,2025-02-23,2025-02-27,77.0,M,17.40,400,App,Follow-up,Yes,3,No-Show,4,1
1017,A0998,P422,C06,2025-05-05,2025-05-16,58.0,M,3.20,850,Walk-In,Consultation,No,1,Attended,11,0
1018,A0999,P509,C05,2025-03-30,2025-04-12,59.0,F,25.56,300,Walk-In,Follow-up,Yes,0,Attended,13,0
1019,A1000,P085,C08,2024-12-03,2025-01-17,38.0,M,10.80,550,App,Follow-up,Yes,2,No-Show,45,1


Then, with a for loop and f-strings, write load_data.sql containing one statement per row, in this pattern (text in single quotes, numbers unquoted): INSERT INTO appointments VALUES ('A0001','P196','C05','2025-01-22',...,0); — and likewise one INSERT INTO clinics VALUES (...) per clinic from clinics_enriched.csv. Write all the clinics statements first, because the appointments table points to them with a foreign key. Before exporting, check and print that appointment_id is unique and that none of the 16 columns has a missing value.

In [ ]:
clinics = pd.read_csv("clinics_enriched.csv")
appointments = data

# 1. checks
print("appointment_id unique:", appointments["appointment_id"].is_unique)
print("number of columns:", appointments.shape[1])
print("missing values per column:")
print(appointments.isna().sum())

# 2.
def sql_value(v):
    if pd.isna(v):
        return "NULL"
    if isinstance(v, str):
        return f"'{v}'"
    return str(v)

# 3. write the file: clinics first, then appointments
with open("load_data.sql", "w") as f:
    for row in clinics.itertuples(index=False):
        values = ", ".join(sql_value(v) for v in row)
        f.write(f"INSERT INTO clinics VALUES ({values});\n")

    for row in appointments.itertuples(index=False):
        values = ", ".join(sql_value(v) for v in row)
        f.write(f"INSERT INTO appointments VALUES ({values});\n")

appointment_id unique: True
number of columns: 16
missing values per column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
